In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets
from torch.optim import SGD

# Chọn cấu hình thiết bị phần cứng
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Tìm nạp tập dữ liệu gốc
data_folder = '~/data/FMNIST'
fmnist = datasets.FashionMNIST(data_folder, download=True, train=True)
tr_images = fmnist.data
tr_targets = fmnist.targets

# 2. Sửa đổi FMNISTDataset để chuẩn hóa hình ảnh đầu vào (chia cho 255)
class FMNISTDataset(Dataset):
    def __init__(self, x, y):
        x = x.float() / 255.0  # QUAN TRỌNG: Chia cho 255 để đưa giá trị pixel về khoảng [0, 1]
        x = x.view(-1, 28 * 28)
        self.x, self.y = x, y
        
    def __getitem__(self, ix):
        x, y = self.x[ix], self.y[ix]
        return x.to(device), y.to(device)
        
    def __len__(self):
        return len(self.x)

# Hàm khởi tạo DataLoader
def get_data():
    train = FMNISTDataset(tr_images, tr_targets)
    trn_dl = DataLoader(train, batch_size=32, shuffle=True)
    return trn_dl

In [ ]:
# 3. Định nghĩa kiến trúc mạng mạng tương tự phần trước
def get_model():
    model = nn.Sequential(
        nn.Linear(28 * 28, 1000),
        nn.ReLU(),
        nn.Linear(1000, 10)
    ).to(device)
    
    loss_fn = nn.CrossEntropyLoss()
    optimizer = SGD(model.parameters(), lr=1e-2)
    return model, loss_fn, optimizer

def train_batch(x, y, model, opt, loss_fn):
    model.train()
    prediction = model(x)
    batch_loss = loss_fn(prediction, y)
    
    batch_loss.backward()
    opt.step()
    opt.zero_grad()
    return batch_loss.item()

@torch.no_grad()
def accuracy(x, y, model):
    model.eval()
    prediction = model(x)
    max_values, argmaxes = prediction.max(-1)
    is_correct = argmaxes == y
    return is_correct.cpu().numpy().tolist()

In [ ]:
# Thực hiện gọi hàm nạp và khởi tạo mô hình
trn_dl = get_data()
model, loss_fn, optimizer = get_model()
losses, accuracies = [], []

# Huấn luyện mô hình qua 5 Epochs
for epoch in range(5):
    print(f"Epoch mã hóa: {epoch}")
    epoch_losses, epoch_accuracies = [], []
    
    for ix, batch in enumerate(iter(trn_dl)):
        x, y = batch
        batch_loss = train_batch(x, y, model, optimizer, loss_fn)
        epoch_losses.append(batch_loss)
    epoch_loss = np.array(epoch_losses).mean()
    
    for ix, batch in enumerate(iter(trn_dl)):
        x, y = batch
        is_correct = accuracy(x, y, model)
        epoch_accuracies.extend(is_correct)
    epoch_accuracy = np.mean(epoch_accuracies)
    
    losses.append(epoch_loss)
    accuracies.append(epoch_accuracy)

# 4. Vẽ biểu đồ hiển thị độ chính xác (Accuracy) và sai số (Loss)
epochs = np.arange(5) + 1
plt.figure(figsize=(20, 5))

# Đồ thị biểu diễn Loss cải thiện qua từng Epoch
plt.subplot(121)
plt.title('Loss value over increasing epochs')
plt.plot(epochs, losses, label='Training Loss')
plt.legend()

# Đồ thị biểu diễn Accuracy cải thiện qua từng Epoch
plt.subplot(122)
plt.title('Accuracy value over increasing epochs')
plt.plot(epochs, accuracies, label='Training Accuracy')
plt.gca().set_yticklabels(['{:.0f}%'.format(x * 100) for x in plt.gca().get_yticks()])
plt.legend()

plt.show()